In [1]:
import openai
print(openai.__version__)

2.15.0


In [4]:
# Load OPENAI API Key
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv("OPENAI_API_KEY")

ImportError: cannot import name 'load_dotenv' from 'dotenv' (c:\Users\my pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\dotenv\__init__.py)

In [6]:
import pandas as pd
import numpy as np
import requests

In [8]:

from openai import OpenAI
import os
import json
import re

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [9]:
# ================================
# Week 3 – LLM Prompt Design
# ================================

SLA_PROMPT_TEMPLATE = """
You are a legal contract analysis assistant specialized in car lease agreements.

Extract the following SLA details from the contract text.
If a value is missing, return null.
Return ONLY valid JSON. No explanation.

Fields:
- interest_rate_apr
- lease_term_months
- monthly_payment
- down_payment
- residual_value
- mileage_allowance
- overage_charge
- early_termination
- purchase_option
- maintenance_responsibility
- warranty_insurance
- penalties
- missing_or_ambiguous_clauses

Contract Text:
\"\"\"
{contract_text}
\"\"\"
"""

In [10]:
# ================================
# Week 3 – OpenAI LLM Extraction
# ================================



def extract_sla_with_gpt(contract_text):
    prompt = SLA_PROMPT_TEMPLATE.format(contract_text=contract_text)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    raw_output = response.output_text.strip()

    match = re.search(r"\{[\s\S]*\}", raw_output)
    if not match:
        return {"error": "No JSON found", "raw_output": raw_output}

    try:
        return json.loads(match.group(0))
    except Exception as e:
        return {"error": str(e), "json_text": match.group(0)}
test_output = extract_sla_with_gpt(df.loc[0,
"extracted_text"])
test_output

NameError: name 'df' is not defined

In [11]:

# ===============================
# Week 3 – Accuracy Evaluation (Sample Only)
# ===============================

# Use only 1 sample contract to avoid rate limits
eval_df = df.head(1).copy()

# Apply LLM extraction (SAFE – only 1 call)
eval_df["llm_sla"] = eval_df["extracted_text"].apply(extract_sla_with_gpt)

eval_df[["llm_sla"]]

NameError: name 'df' is not defined

In [ ]:
# Extract individual SLA fields from LLM output
eval_df["llm_apr"] = eval_df["llm_sla"].apply(lambda x: x.get("interest_rate_apr"))
eval_df["llm_term"] = eval_df["llm_sla"].apply(lambda x: x.get("lease_term_months"))
eval_df["llm_payment"] = eval_df["llm_sla"].apply(lambda x: x.get("monthly_payment"))

eval_df[["llm_apr", "llm_term", "llm_payment"]]
# Ground truth values for sample contract (manual / known values)
eval_df["expected_apr"] = 10.49
eval_df["expected_term"] = 24
eval_df["expected_payment"] = 959

eval_df[[
    "llm_apr", "expected_apr",
    "llm_term", "expected_term",
    "llm_payment", "expected_payment"
]]

In [ ]:
accuracy = {
    "APR Accuracy (%)": (eval_df["llm_apr"] == eval_df["expected_apr"]).mean() * 100,
    "Term Accuracy (%)": (eval_df["llm_term"] == eval_df["expected_term"]).mean() * 100,
    "Payment Accuracy (%)": (eval_df["llm_payment"] == eval_df["expected_payment"]).mean() * 100
}

accuracy

In [ ]:
# ===============================
# Week 4 – VIN Lookup (NHTSA API)
# ===============================

import requests

def fetch_vehicle_details_from_vin(vin):
    """
    Fetch vehicle make, model, year using NHTSA VIN Decode API
    """
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValuesExtended/{vin}?format=json"
    response = requests.get(url, timeout=10)
    data = response.json()

    if not data.get("Results"):
        return None

    result = data["Results"][0]

    return {
        "vin": vin,
        "make": result.get("Make"),
        "model": result.get("Model"),
        "year": result.get("ModelYear")
    }

In [ ]:
def fetch_vehicle_recalls(make, model, year):
    """
    Fetch recall information using NHTSA Recall API
    """
    if not make or not model or not year:
        return []

    url = (
        "https://api.nhtsa.gov/recalls/recallsByVehicle"
        f"?make={make}&model={model}&modelYear={year}"
    )

    response = requests.get(url, timeout=10)
    data = response.json()

    recalls = data.get("results", [])

    return [
        {
            "campaign_number": r.get("NHTSACampaignNumber"),
            "summary": r.get("Summary"),
            "consequence": r.get("Consequence")
        }
        for r in recalls
    ]

In [5]:
test_vin = "1HGCM82633A004352"

vehicle_info = fetch_vehicle_details_from_vin(test_vin)
vehicle_info

NameError: name 'fetch_vehicle_details_from_vin' is not defined

In [ ]:

recalls = fetch_vehicle_recalls(
    vehicle_info["make"],
    vehicle_info["model"],
    vehicle_info["year"]
)

recalls[:2]

In [ ]:
# ===============================
# Week 4 – Combine SLA + Vehicle Data
# ===============================

def build_final_contract_response(contract_id, sla_data, vin):
    vehicle = fetch_vehicle_details_from_vin(vin)

    recalls = []
    if vehicle:
        recalls = fetch_vehicle_recalls(
            vehicle["make"],
            vehicle["model"],
            vehicle["year"]
        )

    return {
        "contract_id": contract_id,
        "sla": sla_data,
        "vehicle": vehicle,
        "recalls": recalls
    }

In [ ]:
# Internal end-to-end test

sample_contract = sla_json_records[0]
sample_vin = "1HGCM82633A004352"

final_response = build_final_contract_response(
    contract_id=sample_contract["contract_id"],
    sla_data=sample_contract["sla"],
    vin=sample_vin
)

final_response